In [1]:
# Allow imports from ../app
import sys, os
sys.path.append("..")

# Load .env (GROQ_API_KEY etc.)
from dotenv import load_dotenv
load_dotenv("../.env", override=True)

# Quick smoke: show key env that must be set
import os
print("ENV            :", os.getenv("ENV"))
print("LLM_MODEL      :", os.getenv("LLM_MODEL"))
print("REDIS_URL      :", os.getenv("REDIS_URL"))
print("VECTOR DIR     :", os.getenv("LOCAL_VECTOR_DIR"))
print("PRODUCTS_SOURCE:", os.getenv("PRODUCTS_SOURCE"))

# Verify imports
from app.settings import settings
print("Settings loaded for env:", settings.env)


python-dotenv could not parse statement starting at line 25


ENV            : dev
LLM_MODEL      : llama-3.3-70b-versatile
REDIS_URL      : redis://localhost:6379/0
VECTOR DIR     : .local_vector_store
PRODUCTS_SOURCE: local
Settings loaded for env: dev


In [2]:
from app.rag.loaders import load_text_folder
from app.rag.retriever import build_or_load_index

docs = load_text_folder("data/raw")
print(f"Found {len(docs)} raw documents.")
_ = build_or_load_index(docs if len(docs) > 0 else None)
print("Vector index is ready (built or loaded).")

print(_)

Found 1 raw documents.


/Users/rahul39duttagmail.com/nexaneura/rag_chatbot/app/rag/retriever.py:37: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  _EMB = HuggingFaceEmbeddings(model_name=settings.embeddings_model)


Vector index is ready (built or loaded).


In [3]:
import redis
from app.settings import settings

r = redis.from_url(settings.redis_url, decode_responses=True)
print("PING →", r.ping())


PING → True


In [4]:
from pprint import pprint

def show(resp):
    print("session_id     :", resp.get("session_id"))
    print("intent         :", resp.get("intent"))
    print("used_rag       :", resp.get("used_rag"))
    print("rag_best_score :", resp.get("rag_best_score"))
    print("redaction      :", resp.get("redaction"))
    print("scope          :", resp.get("scope"))
    print("escalation?    :", "YES" if resp.get("escalation_banner") else "no")
    print("\n--- ANSWER ---\n")
    print(resp.get("answer", "")[:1200])
    print("\n--- PRODUCTS ---")
    for p in resp.get("products", []):
        print("-", p.get("name"))
    print("\n--- MEMORY (recent) ---")
    for t in resp.get("memory_turns", []):
        print(f"{t['role'].upper()}: {t['content'][:160]}")


In [6]:
from app.chains.chat_pipeline import chat_once

res1 = chat_once(
    "Who is rahul dravid?",
    session_id=None  # auto-generate a new session
)
show(res1)

sid = res1["session_id"]  # keep this to continue same conversation
sid


session_id     : e247f957-f8a6-44e7-bd1c-1bcd29f76929
intent         : WELLNESS_INFO
used_rag       : False
rag_best_score : 0.0
redaction      : {'email': 0, 'phone': 0, 'name_hint': 0}
scope          : {'blocked': False, 'matched': []}
escalation?    : no

--- ANSWER ---

Rahul Dravid is a former Indian cricketer and captain of the Indian national team. He is widely regarded as one of the greatest batsmen in the history of cricket. As a new parent, you might be looking for ways to manage stress and find relaxation techniques, just like athletes do. Taking short breaks, practicing deep breathing, and engaging in gentle exercises can help. However, I'm not sure how this information relates to your postpartum wellness. If you have any specific concerns or questions about your well-being, I'm here to listen and provide general guidance. Wellness information only; not medical advice.

— *Wellness information only; not medical advice.*

--- PRODUCTS ---

--- MEMORY (recent) ---
USER: Who i

'e247f957-f8a6-44e7-bd1c-1bcd29f76929'